HW4: Natural Language Processing

Q3
Plain BOW treats every word equally, but words like "the", "is", and "and" show up across
all emotion classes and don't help the model distinguish between them. Removing stop words
shrinks the vocabulary and pushes the model to focus on words that actually carry emotional
meaning.

Q4
TF-IDF normalizes the BOW representation by downweighting words that appear frequently across all documents (low information value) and upweighting words that are rare and distinctive. This means a word like "terrified" that shows up a lot in one class but rarely elsewhere gets a higher weight than something like "feel" which is everywhere.

Q5
Test accuracy: 64.44% ; Macro F1: 0.6466
Per-class F1: jealous=0.646, joyful=0.620, sad=0.615, terrified=0.705
Terrified was easiest to classify. Sad was hardest, getting spread across all three other classes (23→jealous, 27→joyful, 25→terrified).

Misclassified examples:

"she was born premature at home... parents were just praying"
True: sad ; Predicted: terrified
Words like "hard time breathing" read as urgent/alarming. The model has no way of knowing this is a story being retold with sadness rather than live fear.

"my heart speeded so fast!"
True: joyful ; Predicted: sad
A racing heart appears in both excited and anxious contexts. Without knowing this is about seeing a baby ultrasound, it just looks like distress.

"That's perfectly natural... Disappointments like that usually come with some good lessons."
True: sad ; Predicted: joyful
The word "good" and the encouraging tone flipped the prediction. The model can't detect that this is someone consoling another person about something sad. The core problem is that TF-IDF just counts tokens.

Q6
GloVe + MLP scored 60%, worse than SGD. Averaging word vectors loses all context.
"I was not happy" and "I was happy" produce nearly identical embeddings. Joyful took the biggest hit (F1=0.505).

Q7
We used the [CLS] token (index 0 of last_hidden_state, 768-dim) as the sentence embedding. It's designed to summarize the full input sequence, making it a natural fit for classification. BERT came out on top at 69.03% (macro F1: 0.6928) — because it reads words in context, "heart racing" reads differently depending on whether you're scared or excited, and the other methods can't do that.

 Q8.1
- P@1,100: Whether the correct response ranked #1 out of 100 candidates. Directly tests
  if the system picks the right empathetic reply.
- AVG-BLEU: N-gram overlap between generated and reference responses. Known to correlate
  poorly with actual response quality.
- Perplexity: How surprised the model is by the reference text. Measures fluency,
  not empathy.

P@1,100 is the most meaningful metric here- empathetic dialogue is really about selecting
the right response for the moment, and BLEU/PPL don't capture that. BLEU penalizes valid
paraphrases and PPL rewards fluency even if the response is emotionally off.

Q8.2
Models trained with emotion labels consistently beat generic models at the same scale, which
points to emotion-awareness being the main gap. To get closer to human-level performance:
fine-tune on empathetic corpora specifically, add an emotion recognition module that
conditions the response generator, and move away from BLEU/PPL as training targets since
they don't actually measure empathy.

In [18]:
import os, re, collections
import numpy as np
import pandas as pd

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, confusion_matrix, accuracy_score

print('Imports done.')

Imports done.


In [19]:
DATA_DIR = '/Users/gayathripillai/Downloads/empatheticdialogues'  
train_raw = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'), on_bad_lines='skip')
valid_raw = pd.read_csv(os.path.join(DATA_DIR, 'valid.csv'), on_bad_lines='skip')
test_raw  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'),  on_bad_lines='skip')

print('Columns:', train_raw.columns.tolist())
print('Train shape (raw):', train_raw.shape)

Columns: ['conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags']
Train shape (raw): (76668, 8)


In [20]:
#Filtering to 4 target emotions
TARGET_EMOTIONS = {'sad', 'jealous', 'joyful', 'terrified'}

train = train_raw.loc[train_raw['context'].isin(TARGET_EMOTIONS)].copy()
valid = valid_raw.loc[valid_raw['context'].isin(TARGET_EMOTIONS)].copy()
test  = test_raw.loc[test_raw['context'].isin(TARGET_EMOTIONS)].copy()

train = pd.concat([train, valid], ignore_index=True)

print('Train shape:', train.shape)
print('Test shape :', test.shape)
print('Label distribution (train):')
print(train['context'].value_counts())

Train shape: (10609, 8)
Test shape : (720, 8)
Label distribution (train):
context
sad          2863
terrified    2677
joyful       2538
jealous      2531
Name: count, dtype: int64


In [21]:
X_train_raw = train['utterance'].astype(str).tolist()
y_train_raw = train['context'].tolist()

X_test_raw  = test['utterance'].astype(str).tolist()
y_test_raw  = test['context'].tolist()

train_labels_unique = sorted(list(set(y_train_raw)))   
label_mapper = {label: i for i, label in enumerate(train_labels_unique)}
inv_label_mapper = {v: k for k, v in label_mapper.items()}

train_labels_encoded = np.array([label_mapper[l] for l in y_train_raw])
labels_encoded_test  = np.array([label_mapper[l] for l in y_test_raw])

print('Label mapping:', label_mapper)

Label mapping: {'jealous': 0, 'joyful': 1, 'sad': 2, 'terrified': 3}


In [22]:
def remove_punctuation(text_list):
    """Remove punctuation but keep alphanumeric tokens."""
    cleaned = []
    for text in text_list:
        #lowercase, stripping non-alpha characters
        text = text.lower()
        text = re.sub(r'[^a-z\s]', '', text)   
        text = re.sub(r'\s+', ' ', text).strip()
        cleaned.append(text)
    return cleaned

train_data_list_cleaned = remove_punctuation(X_train_raw)
test_data_list_cleaned  = remove_punctuation(X_test_raw)

print('Sample cleaned utterance:', train_data_list_cleaned[0])

Sample cleaned utterance: job interviews always make me sweat bulletscomma makes me uncomfortable in general to be looked at under a microscope like that


In [23]:
#BOW
bow_vectorizer = CountVectorizer()
X_bow_raw = bow_vectorizer.fit_transform(train_data_list_cleaned)
encoding = X_bow_raw.toarray()

#Converting counts to binary (1 = word is present, 0 = word is absent)
encoding[encoding > 0] = 1

print('BOW matrix shape:', encoding.shape)
print('Vocabulary size:', len(bow_vectorizer.vocabulary_))

BOW matrix shape: (10609, 9644)
Vocabulary size: 9644


In [24]:
#Build stop-words list (NLTK + extras that add no emotional signal)
stopwords_list = list(set(stopwords.words('english')))
stopwords_list.extend(['comma', '', 'im', 'ive', 'id', 'hed', 'shed',
                       'thats', 'dont', 'cant', 'couldnt', 'wouldnt',
                       'didnt', 'doesnt', 'isnt', 'wasnt', 'wont',
                       'got', 'get', 'getting', 'said', 'like', 'really',
                       'one', 'going', 'go', 'went', 'come', 'came'])
stopwords_set = set(stopwords_list)

def remove_stopwords(text_list, sw_set):
    cleaned = []
    for text in text_list:
        tokens = text.split()
        tokens = [t for t in tokens if t not in sw_set]
        cleaned.append(' '.join(tokens))
    return cleaned

train_data_stop_removed = remove_stopwords(train_data_list_cleaned, stopwords_set)
test_data_stop_removed  = remove_stopwords(test_data_list_cleaned,  stopwords_set)

print('Sample (stop-words removed):', train_data_stop_removed[0])

Sample (stop-words removed): job interviews always make sweat bulletscomma makes uncomfortable general looked microscope


In [25]:
#BOW after stop-word removal
train_count_vectorizer = CountVectorizer()
X_train_bow = train_count_vectorizer.fit_transform(train_data_stop_removed)

train_one_hot_encoding = X_train_bow.toarray()
train_one_hot_encoding[train_one_hot_encoding > 0] = 1

print('BOW (no stop words) shape:', train_one_hot_encoding.shape)

BOW (no stop words) shape: (10609, 9491)


In [26]:

train_tfidf_transformer = TfidfTransformer(smooth_idf=True, use_idf=True)
train_embedding_tfidf   = train_tfidf_transformer.fit_transform(train_one_hot_encoding)

print('TF-IDF matrix shape:', train_embedding_tfidf.shape)
print('Sample values (first row):', train_embedding_tfidf[0].toarray()[0][:10])

TF-IDF matrix shape: (10609, 9491)
Sample values (first row): [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [27]:
X_train_sgd = train_embedding_tfidf
y_train_sgd = train_labels_encoded

clf_sgd = SGDClassifier(
    loss='modified_huber',  
    max_iter=200,
    tol=1e-3,
    random_state=42,
    class_weight='balanced' 
)
clf_sgd.fit(X_train_sgd, y_train_sgd)

train_preds_sgd = clf_sgd.predict(X_train_sgd)
print('Training accuracy (SGD):', accuracy_score(y_train_sgd, train_preds_sgd))

Training accuracy (SGD): 0.8849090394947686


In [28]:
#Prepare test data with same vocabulary as training 
test_count_vectorizer = CountVectorizer(vocabulary=train_count_vectorizer.vocabulary_)
X_test_bow = test_count_vectorizer.fit_transform(test_data_stop_removed)

test_one_hot_encoding = X_test_bow.toarray()
test_one_hot_encoding[test_one_hot_encoding > 0] = 1

test_tfidf_transformer = TfidfTransformer(smooth_idf=False, use_idf=True)
test_embedding_tfidf   = test_tfidf_transformer.fit_transform(test_one_hot_encoding)
test_preds_sgd = clf_sgd.predict(test_embedding_tfidf)

test_acc_sgd = accuracy_score(labels_encoded_test, test_preds_sgd)
f1_sgd       = f1_score(labels_encoded_test, test_preds_sgd, average=None)
cm_sgd       = confusion_matrix(labels_encoded_test, test_preds_sgd)

print(f'Test accuracy : {test_acc_sgd:.4f}')
print(f'Macro F1 score: {np.mean(f1_sgd):.4f}')
print(f'Per-class F1  : {f1_sgd}')
print('Confusion matrix (rows=true, cols=pred):')
print(pd.DataFrame(cm_sgd,
                   index=train_labels_unique,
                   columns=train_labels_unique))

Test accuracy : 0.6444
Macro F1 score: 0.6466
Per-class F1  : [0.64571429 0.61994609 0.61538462 0.70516717]
Confusion matrix (rows=true, cols=pred):
           jealous  joyful  sad  terrified
jealous        113      31   28         11
joyful          20     115   30         22
sad             23      27  120         25
terrified       11      11   17        116


/Users/gayathripillai/anaconda3/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:1676: RuntimeWarning: divide by zero encountered in divide
  self.idf_ /= df


In [29]:
misclassified_idx = np.where(test_preds_sgd != labels_encoded_test)[0]
print(f'Total misclassified: {len(misclassified_idx)} / {len(labels_encoded_test)}')
print()

#Printing 10 misclassified examples
for i in misclassified_idx[:10]:
    true_label = inv_label_mapper[labels_encoded_test[i]]
    pred_label = inv_label_mapper[test_preds_sgd[i]]
    utterance  = X_test_raw[i]
    print(f'Utterance : {utterance}')
    print(f'True label: {true_label}  |  Predicted: {pred_label}')
    print('-' * 60)

Total misclassified: 256 / 720

Utterance : she was born premature at home_comma_ she had hard time breathing on her own but instead of taking her to the doctor parents were just praying
True label: sad  |  Predicted: terrified
------------------------------------------------------------
Utterance : yes! And i do believe in God and prayers but goodness gracious please take your children to the hospital and let God heal them THROUGH doctors
True label: sad  |  Predicted: terrified
------------------------------------------------------------
Utterance : She was around 11_comma_ so she took it very hard. I went ahead and let her stay home from school_comma_ as I knew she wouldn't do well that day.
True label: sad  |  Predicted: jealous
------------------------------------------------------------
Utterance : One of the saddest things to me is when people underestimate what they can do and/or are capable of.
True label: sad  |  Predicted: jealous
--------------------------------------------

In [30]:
import gensim
import gensim.downloader as api
print('Downloading GloVe 100d model (this may take a moment)...')
w2v_model = api.load('glove-wiki-gigaword-100')   # 100-dim GloVe vectors
EMBED_DIM = 100
print('Model loaded. Vocabulary size:', len(w2v_model.key_to_index))

Model loaded. Vocabulary size: 400000


In [31]:
def sentence_to_avg_embedding(sentences, model, dim):
    embeddings = []
    for sent in sentences:
        tokens = nltk.word_tokenize(sent.lower())
        vecs = [model[t] for t in tokens if t in model]
        if vecs:
            embeddings.append(np.mean(vecs, axis=0))
        else:
            embeddings.append(np.zeros(dim))
    return np.array(embeddings)

X_train_w2v = sentence_to_avg_embedding(train_data_stop_removed, w2v_model, EMBED_DIM)
X_test_w2v  = sentence_to_avg_embedding(test_data_stop_removed,  w2v_model, EMBED_DIM)

print('Train embeddings shape:', X_train_w2v.shape)
print('Test  embeddings shape:', X_test_w2v.shape)

Train embeddings shape: (10609, 100)
Test  embeddings shape: (720, 100)


In [32]:
#MLP Classifier on Word2Vec embeddings
mlp_w2v = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation='relu',
    max_iter=300,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)
mlp_w2v.fit(X_train_w2v, train_labels_encoded)

test_preds_w2v = mlp_w2v.predict(X_test_w2v)
test_acc_w2v   = accuracy_score(labels_encoded_test, test_preds_w2v)
f1_w2v         = f1_score(labels_encoded_test, test_preds_w2v, average=None)
cm_w2v         = confusion_matrix(labels_encoded_test, test_preds_w2v)

print(f'Test accuracy : {test_acc_w2v:.4f}')
print(f'Macro F1 score: {np.mean(f1_w2v):.4f}')
print(f'Per-class F1  : {f1_w2v}')
print('Confusion matrix:')
print(pd.DataFrame(cm_w2v,
                   index=train_labels_unique,
                   columns=train_labels_unique))

Test accuracy : 0.6000
Macro F1 score: 0.5983
Per-class F1  : [0.61875    0.50455927 0.59764706 0.67213115]
Confusion matrix:
           jealous  joyful  sad  terrified
jealous         99      30   29         25
joyful          16      83   60         28
sad             14      19  127         35
terrified        8      10   14        123


In [33]:
import torch
from transformers import DistilBertTokenizer, DistilBertModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

tokenizer_bert = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model_bert      = DistilBertModel.from_pretrained('distilbert-base-uncased')
model_bert.eval()
model_bert.to(device)
print('DistilBERT loaded.')

Using device: cpu


Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing DistilBertModel: ['vocab_projector.weight', 'vocab_transform.weight', 'vocab_transform.bias', 'vocab_layer_norm.weight', 'vocab_layer_norm.bias', 'vocab_projector.bias']
- This IS expected if you are initializing DistilBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DistilBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


DistilBERT loaded.


In [34]:
def get_bert_embeddings(sentences, tokenizer, model, device, batch_size=32):
    all_embeddings = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(sentences), batch_size):
            batch = sentences[start:start + batch_size]
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors='pt'
            )
            input_ids      = encoded['input_ids'].to(device)
            attention_mask = encoded['attention_mask'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            # last_hidden_state: (batch, seq_len, 768)
            # We take the [CLS] token (index 0) as the sentence representation
            cls_embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(cls_embedding)

    return np.vstack(all_embeddings)

X_train_bert = get_bert_embeddings(X_train_raw, tokenizer_bert, model_bert, device)
X_test_bert  = get_bert_embeddings(X_test_raw,  tokenizer_bert, model_bert, device)

print('BERT train shape:', X_train_bert.shape)
print('BERT test  shape:', X_test_bert.shape)

BERT train shape: (10609, 768)
BERT test  shape: (720, 768)


In [35]:
mlp_bert = MLPClassifier(
    hidden_layer_sizes=(512, 256),
    activation='relu',
    max_iter=300,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)
mlp_bert.fit(X_train_bert, train_labels_encoded)

test_preds_bert = mlp_bert.predict(X_test_bert)
test_acc_bert   = accuracy_score(labels_encoded_test, test_preds_bert)
f1_bert         = f1_score(labels_encoded_test, test_preds_bert, average=None)
cm_bert         = confusion_matrix(labels_encoded_test, test_preds_bert)

print(f'Test accuracy : {test_acc_bert:.4f}')
print(f'Macro F1 score: {np.mean(f1_bert):.4f}')
print(f'Per-class F1  : {f1_bert}')
print('Confusion matrix:')
print(pd.DataFrame(cm_bert,
                   index=train_labels_unique,
                   columns=train_labels_unique))

Test accuracy : 0.6903
Macro F1 score: 0.6928
Per-class F1  : [0.66852368 0.67029973 0.66836735 0.76397516]
Confusion matrix:
           jealous  joyful  sad  terrified
jealous        120      30   24          9
joyful          23     123   26         15
sad             26      18  131         20
terrified        7       9   16        123


## Summary Comparison

In [36]:
summary = pd.DataFrame({
    'Method'      : ['SGD (TF-IDF BOW)', 'MLP (Word2Vec/GloVe)', 'MLP (DistilBERT CLS)'],
    'Test Accuracy': [test_acc_sgd, test_acc_w2v, test_acc_bert],
    'Macro F1'    : [np.mean(f1_sgd), np.mean(f1_w2v), np.mean(f1_bert)]
})
print(summary.to_string(index=False))

              Method  Test Accuracy  Macro F1
    SGD (TF-IDF BOW)       0.644444  0.646553
MLP (Word2Vec/GloVe)       0.600000  0.598272
MLP (DistilBERT CLS)       0.690278  0.692791
